# Step 1 - Rendering B interface-calibration transfer curves

This compares A-trained -> B with seed-0 initialization -> B at identical cumulative B budgets. Budget zero is explicitly diagnostic; calibrated points measure transfer. Add the successful dense seed-0 notebook output as a Kaggle Input before running. Scientific differences are printed without capability thresholds.

Render this SHA template after committing with `python step1/kaggle/render_preflight_notebook.py --notebook step1/kaggle/step1_rendering_b_transfer.ipynb --commit <final-40-char-sha>`.

In [ ]:
from pathlib import Path
import os
import subprocess
import shutil
import urllib.request
import sys

REPO_URL = 'https://github.com/escher-bach/actuallybuildingstuff.git'
GIT_COMMIT = '__FINAL_COMMIT_SHA__'
CONFIG_REL = 'step1/configs/kaggle/t4x2_rendering_b_transfer_seed0.toml'
if not (len(GIT_COMMIT) == 40 and all(c in '0123456789abcdef' for c in GIT_COMMIT)):
    raise RuntimeError('Generate this template with render_preflight_notebook.py after committing the transfer runner.')
WORKING = Path('/kaggle/working')
SOURCE = WORKING / 'actuallybuildingstuff'
PROJECT = SOURCE / 'baby-llm-foundations'
OUTPUT = WORKING / 'rendering-b-transfer-seed0'
assert not SOURCE.exists(), f'fresh batch session required; already exists: {SOURCE}'


In [ ]:
env = os.environ.copy()
env.update({'GIT_TERMINAL_PROMPT': '0', 'PYTHONUNBUFFERED': '1', 'PIP_DISABLE_PIP_VERSION_CHECK': '1', 'WANDB_MODE': 'disabled', 'TOKENIZERS_PARALLELISM': 'false'})
subprocess.run(['git', 'clone', REPO_URL, str(SOURCE)], check=True, env=env)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', GIT_COMMIT], check=True, env=env)
resolved = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True, env=env).strip()
assert resolved == GIT_COMMIT, (resolved, GIT_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements-kaggle.txt')], check=True, env=env)
if not shutil.which('cargo'):
    rustup = WORKING / 'rustup-init'
    urllib.request.urlretrieve('https://static.rust-lang.org/rustup/dist/x86_64-unknown-linux-gnu/rustup-init', rustup)
    rustup.chmod(0o755)
    subprocess.run([str(rustup), '-y', '--profile', 'minimal', '--default-toolchain', '1.85.0'], check=True, env=env)
    env['PATH'] = str(Path.home() / '.cargo/bin') + os.pathsep + env['PATH']
assert shutil.which('cargo', path=env['PATH']), 'pinned Rust installation did not provide Cargo'
subprocess.run([sys.executable, '-m', 'maturin', 'build', '--release', '--manifest-path', str(PROJECT / 'step1/crates/world-py/Cargo.toml')], cwd=str(PROJECT / 'step1'), check=True, env=env)
wheels = sorted((PROJECT / 'step1/target/wheels').glob('world_py-*.whl'))
assert wheels, 'world_py wheel was not built'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', str(wheels[-1])], check=True, env=env)


In [ ]:
# Kaggle exposes a previous notebook output below /kaggle/input. Resolve it
# by exact dense report identity, never by directory naming alone.
sys.path.insert(0, str(PROJECT / 'step1/python'))
from step1_experiments.transfer import load_transfer_config, locate_dense_source
transfer_config = load_transfer_config(PROJECT / CONFIG_REL)
try:
    DENSE_SOURCE = locate_dense_source([Path('/kaggle/input')], transfer_config['source'])
except RuntimeError as error:
    raise RuntimeError('Dense seed-0 output is absent. In Kaggle choose Add Input, select the successful dense seed-0 notebook output, then rerun. ' + str(error)) from error
print('Using exact dense source:', DENSE_SOURCE)


In [ ]:
cmd = ['torchrun', '--standalone', '--nproc_per_node=2', '-m', 'step1_experiments.transfer', '--config', str(PROJECT / CONFIG_REL), '--source-run', str(DENSE_SOURCE), '--output-dir', str(OUTPUT)]
completed = subprocess.run(cmd, cwd=str(PROJECT / 'step1/python'), env=env, check=False)
if completed.returncode != 0:
    raise RuntimeError(f'Rendering B transfer run failed with exit code {completed.returncode}; inspect {OUTPUT}')


In [ ]:
import hashlib
import json
import math
import tomllib

report_path = OUTPUT / 'rendering_b_transfer_report.json'
assert report_path.is_file(), report_path
report = json.loads(report_path.read_text())
config = tomllib.loads((PROJECT / CONFIG_REL).read_text())
config_hash = hashlib.sha256(json.dumps(config, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
budgets = [0, 31, 92, 306, 916, 3052]
tokens = [0, 1015808, 3014656, 10027008, 30015488, 100007936]
plan = report['plan']
assert report['contract'] == 'step1_rendering_b_transfer_v1', report
assert report['experiment_config_sha256'] == config_hash, report
assert report['source']['git_sha'] == config['source']['git_sha'], report
assert report['source']['config_hash'] == config['source']['config_hash'], report
assert report['source']['model_state_sha256'] == config['source']['model_state_sha256'], report
assert report['source']['prior_a_training_cost']['token_budget'] == 100007936, report
assert plan['world_size'] == 2 and plan['per_device_sequences'] == 4, plan
assert plan['context_length'] == 2048 and plan['gradient_accumulation_steps'] == 2, plan
assert plan['nominal_global_input_tokens_per_update'] == 32768, plan
assert plan['budgets_updates'] == budgets and plan['budgets_nominal_global_input_tokens'] == tokens, plan
assert plan['full_b_reference_included'] is True, plan
calibration = report['datasets']['calibration']
assert calibration['rendering'] == 'b' and calibration['seed'] == 23260811 and calibration['episodes'] == 32768, calibration
assert calibration['prefix_equivalence']['exact'] is True, calibration
assert calibration['prefix_equivalence']['expected_data_sha256'] == config['source']['calibration_data_sha256'], calibration
assert Path(calibration['path']).is_file(), calibration
assert hashlib.sha256(Path(calibration['path']).read_bytes()).hexdigest() == calibration['data_sha256'], calibration
heldout = report['datasets']['held_out_evaluation_worlds']
assert heldout == {'rendering': 'b', 'seed': 21260811, 'episodes': 1024, 'matched_across_arms_and_budgets': True, 'variants': ['irreversible', 'reversible_control']}, heldout
assert set(report['arms']) == {'a_trained', 'init'}, report
for arm_name, arm in report['arms'].items():
    assert arm['root_seed'] == 20260811 and arm['ranks_finished'] == [0, 1], arm
    assert arm['calibration_data_sha256'] == calibration['data_sha256'], arm
    assert [point['budget_updates'] for point in arm['curve']] == budgets, arm
    assert [point['b_calibration_nominal_global_input_tokens'] for point in arm['curve']] == tokens, arm
    assert arm['curve'][0]['point_label'] == 'zero_shot_interface_diagnostic_not_transfer', arm
    assert arm['curve'][-1]['point_label'] == 'full_b_reference', arm
    for point in arm['curve']:
        exact = point['serialization']
        assert exact['exact'] is True and exact['expected_state_sha256'] == exact['actual_state_sha256'], exact
        assert set(point['metrics']) == {'irreversible', 'reversible_control'}, point
        for metrics in point['metrics'].values():
            assert all(value is None or math.isfinite(value) for value in metrics.values()), metrics
        if point['budget_updates']:
            assert Path(point['artifact']).name == f"checkpoint-{point['budget_updates']}" and Path(point['artifact']).is_dir(), point
assert report['scientific_acceptance_policy'] == 'diagnostic learning curves; no capability threshold', report
assert [point['budget_updates'] for point in report['paired_diagnostics']] == budgets, report
assert all(point['definition'] == 'a_trained_minus_init; paired diagnostic only' for point in report['paired_diagnostics']), report
curves = {arm: [{
    'budget_tokens': p['b_calibration_nominal_global_input_tokens'],
    'label': p['point_label'],
    'metrics': p['metrics'],
} for p in value['curve']] for arm, value in report['arms'].items()}
print(json.dumps({'curves': curves, 'paired_diagnostics': report['paired_diagnostics']}, indent=2, sort_keys=True))
